# YOLO11 Training - Tea Disease Detection

Notebook ini khusus untuk training YOLO11 di folder terpisah dari training YOLOv8 lama.

Tujuan:
- Training YOLO11 dengan dataset yang sama seperti YOLOv8 lama
- Setting training dibuat sama seperti notebook lama
- Menyimpan metrics `precision`, `recall`, `mAP50`, dan `mAP50-95`
- Menyimpan `yolo11m_best.pt` dan `best_overall.pt`


## CELL 1 — Install Dependencies

In [ ]:
print("Installing required packages...")
import subprocess
import sys

# Sama seperti notebook lama: siapkan PyTorch + Ultralytics + Roboflow.
# Ultralytics di-upgrade agar model baru seperti YOLO11/YOLO12/YOLO26 tersedia.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.4.1", "torchvision==0.19.1",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade", "ultralytics", "roboflow", "pyyaml", "pandas"
], check=True)

import torch
import ultralytics
print(f"\n✅ Ultralytics: {ultralytics.__version__}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## CELL 2 — Configuration

In [ ]:
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# MODEL YANG AKAN DITRAIN
# Memakai varian medium agar sebanding dengan training lama: yolov8m.pt
# Kalau Kaggle terlalu lama, ubah menjadi yolo11n.pt.
# ─────────────────────────────────────────────────────────────
MODELS_TO_TRAIN = [
    "yolo11m.pt",
]

# ─────────────────────────────────────────────────────────────
# SETTING TRAINING — DISAMAKAN DENGAN NOTEBOOK LAMA
# ─────────────────────────────────────────────────────────────
EPOCHS = 100
IMGSZ = 640
BATCH = 8
PATIENCE = 20
OPTIMIZER = "SGD"
LR0 = 0.01

# Roboflow dataset Anda — sama seperti notebook lama
ROBOFLOW_API_KEY = "WPrpJznPxK5BhArjmMUg"
ROBOFLOW_WORKSPACE = "dyl-hgadx"
ROBOFLOW_PROJECT = "tehobject"
ROBOFLOW_VERSION = 11

base_dir = Path('/kaggle/working/tea_disease_multi_yolo')
dirs = {
    'dataset': base_dir / 'dataset',
    'runs': base_dir / 'runs',
    'exports': Path('/kaggle/working/benchmark_exports'),
}

for name, path in dirs.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"✅ {name}: {path}")

print("\n🎯 Model yang akan ditraining:")
for m in MODELS_TO_TRAIN:
    print(f"- {m}")

print("\n⚙️ Setting training:")
print(f"epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, patience={PATIENCE}, optimizer={OPTIMIZER}, lr0={LR0}")

## CELL 3 — Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
import shutil
from pathlib import Path

# Hapus folder lama agar Roboflow tidak skip download
# PENTING: jangan pre-create folder, biarkan Roboflow yang buat
if dirs['dataset'].exists():
    shutil.rmtree(dirs['dataset'])
    print(f'🗑️  Folder lama dihapus: {dirs["dataset"]}')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset_version = project.version(ROBOFLOW_VERSION)

print('📥 Downloading dataset dari Roboflow...')
dataset = dataset_version.download('yolov8', location=str(dirs['dataset']), overwrite=True)
print(f'✅ Download selesai')
print(f'   dataset.location = {getattr(dataset, "location", "N/A")}')

# Debug: tampilkan semua file di bawah /kaggle/working
print('\n📂 Semua data.yaml yang ditemukan di /kaggle/working:')
all_yaml = list(Path('/kaggle/working').rglob('data.yaml'))
for y in all_yaml:
    print(f'   {y}')

if not all_yaml:
    print('   ⚠️ TIDAK ADA data.yaml — tampilkan semua file:')
    for item in sorted(Path('/kaggle/working').iterdir())[:20]:
        print(f'   {item.name}/')

# Cari data.yaml: urutan prioritas
yaml_files = list(dirs['dataset'].rglob('data.yaml'))

if not yaml_files and hasattr(dataset, 'location'):
    yaml_files = list(Path(dataset.location).rglob('data.yaml'))

if not yaml_files:
    yaml_files = list(Path('/kaggle/working').rglob('data.yaml'))

if not yaml_files:
    raise FileNotFoundError('data.yaml tidak ditemukan. Lihat debug output di atas.')

data_yaml = str(yaml_files[0])
print(f'\n✅ data.yaml ditemukan: {data_yaml}')

## CELL 4 — Train dan Validate Semua Model

In [ ]:
from ultralytics import YOLO
import torch
import pandas as pd
import shutil
from pathlib import Path

def safe_model_name(model_name: str) -> str:
    return model_name.replace('.pt', '').replace('-', '_').replace('/', '_')

def get_metric(metrics, key, default=None):
    try:
        return metrics.results_dict.get(key, default)
    except Exception:
        return default

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Training on: {device}")

summary_rows = []
failed_models = []

for model_ckpt in MODELS_TO_TRAIN:
    short_name = safe_model_name(model_ckpt)
    run_name = short_name + "_tea_disease"

    print("\n" + "=" * 80)
    print(f"🚀 START TRAINING: {model_ckpt}")
    print("=" * 80)

    try:
        model = YOLO(model_ckpt)
        model.train(
            data=data_yaml,
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=BATCH,
            device=device,
            project=str(dirs['runs']),
            name=run_name,
            patience=PATIENCE,
            save=True,
            verbose=True,
            optimizer=OPTIMIZER,
            lr0=LR0,
            exist_ok=True,
        )

        run_dir = dirs['runs'] / run_name
        best_path = run_dir / 'weights' / 'best.pt'
        last_path = run_dir / 'weights' / 'last.pt'

        if not best_path.exists():
            raise FileNotFoundError(f"best.pt tidak ditemukan: {best_path}")

        print(f"\n📊 Validating: {model_ckpt}")
        eval_model = YOLO(str(best_path))
        metrics = eval_model.val(data=data_yaml, device=device, imgsz=IMGSZ, verbose=False)

        row = {
            'model': model_ckpt,
            'run_name': run_name,
            'epochs': EPOCHS,
            'imgsz': IMGSZ,
            'batch': BATCH,
            'optimizer': OPTIMIZER,
            'lr0': LR0,
            'precision': get_metric(metrics, 'metrics/precision(B)'),
            'recall': get_metric(metrics, 'metrics/recall(B)'),
            'mAP50': get_metric(metrics, 'metrics/mAP50(B)'),
            'mAP50_95': get_metric(metrics, 'metrics/mAP50-95(B)'),
            'best_pt': str(best_path),
        }
        summary_rows.append(row)

        export_dir = dirs['exports'] / short_name
        export_dir.mkdir(parents=True, exist_ok=True)

        shutil.copy(best_path, export_dir / f"{short_name}_best.pt")
        shutil.copy(best_path, Path('/kaggle/working') / f"{short_name}_best.pt")

        if last_path.exists():
            shutil.copy(last_path, export_dir / f"{short_name}_last.pt")

        for artifact in ['results.csv', 'results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png']:
            src = run_dir / artifact
            if src.exists():
                shutil.copy(src, export_dir / f"{short_name}_{artifact}")

        print(f"✅ DONE: {model_ckpt}")

    except Exception as e:
        print(f"❌ FAILED: {model_ckpt}")
        print(f"   Error: {e}")
        failed_models.append({'model': model_ckpt, 'error': str(e)})

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_df = summary_df.sort_values('mAP50_95', ascending=False)
    summary_path = Path('/kaggle/working/yolo_benchmark_summary.csv')
    summary_df.to_csv(summary_path, index=False)
    print("\n" + "=" * 80)
    print("🏆 BENCHMARK SUMMARY")
    print("=" * 80)
    display(summary_df)
    print(f"\n✅ Summary saved: {summary_path}")
else:
    print("❌ Tidak ada model yang berhasil selesai training.")

if failed_models:
    failed_df = pd.DataFrame(failed_models)
    failed_path = Path('/kaggle/working/yolo_benchmark_failed.csv')
    failed_df.to_csv(failed_path, index=False)
    print("\n⚠️ Model yang gagal:")
    display(failed_df)
    print(f"⚠️ Failed log saved: {failed_path}")

## CELL 5 — Pilih Model Terbaik

In [ ]:
import pandas as pd
import shutil
from pathlib import Path

summary_path = Path('/kaggle/working/yolo_benchmark_summary.csv')

if summary_path.exists():
    df = pd.read_csv(summary_path)
    print("📊 Ranking berdasarkan mAP50-95:")
    display(df)

    best_row = df.iloc[0]
    best_model_name = best_row['model']
    best_pt = Path(best_row['best_pt'])

    print("\n🏆 MODEL TERBAIK:")
    print(f"Model     : {best_model_name}")
    print(f"mAP50     : {best_row['mAP50']}")
    print(f"mAP50-95  : {best_row['mAP50_95']}")
    print(f"best.pt   : {best_pt}")

    if best_pt.exists():
        shutil.copy(best_pt, Path('/kaggle/working/best_overall.pt'))
        print("\n✅ best_overall.pt saved to /kaggle/working/best_overall.pt")

    report = Path('/kaggle/working/BENCHMARK_RESULT.txt')
    report.write_text(
        "YOLO Benchmark Result - Tea Disease Detection\n"
        "================================================\n\n"
        f"Best model : {best_model_name}\n"
        f"mAP50      : {best_row['mAP50']}\n"
        f"mAP50-95   : {best_row['mAP50_95']}\n"
        f"Precision  : {best_row['precision']}\n"
        f"Recall     : {best_row['recall']}\n\n"
        "Full table saved in yolo_benchmark_summary.csv\n"
    )
    print(f"✅ Report saved: {report}")
else:
    print("❌ Summary file belum ada. Jalankan cell training dulu.")

## Output yang Perlu Di-download

Setelah Kaggle selesai, buka tab **Output** dan download:

- `yolo_benchmark_summary.csv`
- `BENCHMARK_RESULT.txt`
- `best_overall.pt`
- `yolo11m_best.pt`

Test lokal contoh:

```bash
python3 test_local.py --model best_overall.pt --input Tehbaru.mp4 --output hasil_yolo11.mp4
```
